### Runnables 구조 검토


In [1]:
from dotenv import load_dotenv

load_dotenv()

from langchain_teddynote import logging

logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [2]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    # 텍스트 데이터로부터 FAISS 벡터 저장소를 생성
    ["Teddy is an AI engineer who loves programming!"],
    embedding= OpenAIEmbeddings()
)

# 벡터 저장소를 기반으로 retriever 생성
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}  

Question: {question}"""

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(model="gpt-4o-mini")

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt | model | StrOutputParser()
)

- 그래프 구성 확인

In [3]:
# 체인의 그래프에서 노드를 가져옴
chain.get_graph().nodes

# 엣지를 가져옴
chain.get_graph().edges

[Edge(source='0e102891e90241ceb3d41e353da26f51', target='f673ca749f0c4aa6859a59ca61a042fb', data=None, conditional=False),
 Edge(source='f673ca749f0c4aa6859a59ca61a042fb', target='646702a1c828406aaf56b5d21f5cbe25', data=None, conditional=False),
 Edge(source='0e102891e90241ceb3d41e353da26f51', target='18b5396b623e467d949c15b1dea44cd3', data=None, conditional=False),
 Edge(source='18b5396b623e467d949c15b1dea44cd3', target='646702a1c828406aaf56b5d21f5cbe25', data=None, conditional=False),
 Edge(source='646702a1c828406aaf56b5d21f5cbe25', target='076b897ee25e4e1aa49340dbbae1d6df', data=None, conditional=False),
 Edge(source='076b897ee25e4e1aa49340dbbae1d6df', target='f92d04d7ec6949c4a0a4ab8ec889001f', data=None, conditional=False),
 Edge(source='1716fbfb96174ab9898f36734fe7bc18', target='9b27e2b75d8d40239345202c2201da1d', data=None, conditional=False),
 Edge(source='f92d04d7ec6949c4a0a4ab8ec889001f', target='1716fbfb96174ab9898f36734fe7bc18', data=None, conditional=False)]

In [4]:
# 그래프 출력
chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                             *                  

In [5]:
# 프롬프트 가져오기
# 체인에서 중요한 부분은 사용되는 프롬프트
chain.get_prompts()

[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}  \n\nQuestion: {question}'), additional_kwargs={})])]